## Merge Author Profiles — Both-No-ORCID Rich-Name, Size-3 to Size-9 Groups

Phase 3c of the author-merge initiative. Extends the rich-name + signal recipe to **size-3 through size-9 groups containing zero ORCID-bearing profiles** — the both-no-ORCID extension of Phase 3b's one-ORCID-anchor pass.

Companion to `MergeAuthorsOrcidCollision` (Phase 1), `MergeAuthorsRichName` (Phase 2 — size-2 both-no-ORCID + one-ORCID-anchor), and `MergeAuthorsOrcidAnchorMulti` (Phase 3b — size-3–9 one-ORCID-anchor). This pass picks up the size-3+ extension of the both-no-ORCID case — fragmentation patterns where multiple no-ORCID profiles for the same human accumulate without an ORCID profile to anchor them.

**Scope:** size-3 through size-9 groups, zero ORCID-bearing profiles. Audit (2026-05-01, n=150 across two rounds): 144/150 = 96% precision (Option B group-anchor signal floor), 0 clear false positives at sz=3, residual marginals concentrated in Brazilian common-surname + polluted-affiliation and Korean common-name at large sizes.

**Volume estimate:** ~33,856 candidate groups → ~88,561 loser profiles.

**Precision guards:**
1. **Tightened rich-name only.** Parsed `first` and `last` must each be ≥ 2 chars **after stripping any trailing period** (`LENGTH(REGEXP_REPLACE(p_first, '[.]$', '')) >= 2`). Without the strip-then-measure, parsed `"S."` (length 2 with dot) passes and collapses initials-first cases like `S. Hernández Gómez` / `A. Eve Miller` into single groups — a failure mode Phase 3b's ORCID anchor masked but Phase 3c is exposed to. Parsed `middle` must contain at least one word ≥ 3 chars.
2. **Dominant raw_name majority (≥ 50%).** Same as Phase 2 / 3b — drops profiles with stray contaminating raw_names that compete with the canonical.
3. **Full_name self-consistency.** Same as Phase 3b — every profile's `openalex_authors.full_name` (lowercased + ASCII-folded) must contain the parsed `last` AND the first ≥ 3-char word of parsed `middle` as whole tokens.
4. **Group size 3–9.** Same as Phase 3b.
5. **Sink cap.** `works_count <= 5,000` on every profile.
6. **Zero ORCID-bearing profiles per group.** The both-no-ORCID twist on Phase 3b's "exactly one ORCID" condition. ORCID-anchored size-3+ groups are already handled by Phase 3b.
7. **Per-pair precision signal required:** every `(winner, straggler)` pair must independently share inst-overlap OR ≥ 1 coauthor (excluding other group members). Same as Phase 3b.
8. **Group-anchor signal floor (Option B, new in Phase 3c).** At least ONE pair in the group must have `inst_overlap = TRUE` OR `n_shared_coauthors >= 5`. Required because, without an ORCID anchor, weak-signal-everywhere groups collapse multiple distinct humans whose romanized names happen to coincide (canonical example: Korean three-token names like `Kyung Kwang Lee` sz=9 with all pair COA=1–3 and no inst — drops out under this floor). The floor preserves groups that have at least one strong-signal pair, which the v2 audit confirmed are reliably the same human.

**Winner selection:** no ORCID anchor available. Winner = `max(works_count)` → `min(created_date)` → `min(author_id)`. The full_name self-consistency guard (#3) defends against the wrong-winner-selection failure mode by requiring every group member's `full_name` to be parser-compatible with the dominant raw name; profiles where `full_name` says one human but attributions say another are dropped at the group level.

**Loser handling:** *No DELETE.* Same MERGE-then-soft-zero-via-CreateAuthors approach as Phases 1 / 2 / 3b. `work_authors.author_id` rewrites loser → winner; CreateAuthors drains loser `works_count` to 0 on next refresh; ES sees an updated zero-works document instead of a missing-id orphan.

**Rollback plan:** No automatic rollback. Audit log `openalex.authors.author_merge_log_richname_multi` captures the winner/loser pairing, parsed name, and signal flags; sufficient to reconstruct any loser via `STILL_UNMATCHED` + `MatchAuthors`.

**Known residual failure modes (out of scope for Phase 3c):**
- **Brazilian common-surname + polluted affiliation** (e.g. multiple "Bruno Sidnei da Silva" profiles each with a grab-bag of unrelated institutions). The anchor signal floor passes these via weak inst-overlap on a generic affiliation; ~3% incidence in the n=100 audit. A future Phase 3d targeting common-surname affiliation cleanup would address this.
- **Korean common-name at size 9** (e.g. `Byong Wan Kim` sz=9). Anchor floor catches the worst cases; residuals have at least one COA≥5 pair, but per-pair signal can still be thin across the full group.


### Cell 1: Build merge candidates table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.merge_candidates_richname_multi AS
WITH
all_profiles AS (
  SELECT id AS author_id, orcid, full_name, display_name, block_key,
         works_count, cited_by_count, created_date,
         TRANSFORM(COALESCE(last_known_institutions, ARRAY()), x -> x.id) AS lki_ids,
         TRANSFORM(COALESCE(affiliations,            ARRAY()), x -> x.institution.id) AS aff_ids
  FROM openalex.authors.openalex_authors
),
ra_counts AS (
  SELECT wa.author_id, wa.raw_author_name, COUNT(*) AS n_works
  FROM openalex.works.work_authors wa
  JOIN all_profiles n ON n.author_id = wa.author_id
  WHERE wa.raw_author_name IS NOT NULL AND TRIM(wa.raw_author_name) <> ''
  GROUP BY wa.author_id, wa.raw_author_name
),
ra_dominant AS (
  -- Dominant raw_name per profile, weighted by work_authors row count.
  -- Require >= 50% majority to drop profiles where stray contaminating raw_names
  -- compete with the canonical (Phase 1 Rappard pollution case).
  SELECT author_id, raw_author_name
  FROM (
    SELECT author_id, raw_author_name, n_works,
           SUM(n_works) OVER (PARTITION BY author_id) AS total_works,
           ROW_NUMBER() OVER (PARTITION BY author_id
                              ORDER BY n_works DESC, raw_author_name ASC) AS rn
    FROM ra_counts
  )
  WHERE rn = 1 AND n_works * 2 >= total_works
),
parsed_dominant AS (
  -- Resolve dominant raw_name to (first, middle, last); tightened rich-name filter:
  --  * first/last >= 2 chars AFTER stripping trailing period (defends against
  --    initials-first cases like "S. Hernández Gómez" -- Phase 3b's ORCID anchor
  --    masked this; Phase 3c is exposed without that protection).
  --  * middle must contain at least one word of length >= 3.
  SELECT rd.author_id,
         REGEXP_REPLACE(an.parsed_name.first,  '[.]$', '') AS p_first,
         REGEXP_REPLACE(an.parsed_name.middle, '[.]$', '') AS p_middle,
         REGEXP_REPLACE(an.parsed_name.last,   '[.]$', '') AS p_last
  FROM ra_dominant rd
  JOIN openalex.authors.author_names an ON an.raw_author_name = rd.raw_author_name
  WHERE LENGTH(REGEXP_REPLACE(an.parsed_name.first, '[.]$', '')) >= 2
    AND LENGTH(REGEXP_REPLACE(an.parsed_name.last,  '[.]$', '')) >= 2
    AND SIZE(FILTER(
          SPLIT(REGEXP_REPLACE(an.parsed_name.middle, '[.]', ''), ' +'),
          x -> LENGTH(x) >= 3
        )) >= 1
),
profile_with_dom AS (
  SELECT n.*, p.p_first, p.p_middle, p.p_last,
         (n.orcid IS NOT NULL AND n.orcid <> '') AS has_orcid
  FROM all_profiles    n
  JOIN parsed_dominant p ON p.author_id = n.author_id
),
profile_consistent AS (
  -- Full_name self-consistency guard (same as Phase 3b).
  SELECT pwd.*,
    CASE
      WHEN full_name IS NULL THEN FALSE
      WHEN INSTR(
        CONCAT(' ', LOWER(REGEXP_REPLACE(
          TRANSLATE(full_name,
            'áéíóúüñçÁÉÍÓÚÜÑÇàèìòùâêîôûÀÈÌÒÙÂÊÎÔÛãõÃÕıİğĞşŞçÇöÖüÜæøłÆØŁčćžšŠ',
            'aeiouuncAEIOUUNCaeiouaeiouAEIOUAEIOUaoaoiIgGsScCoOuUaolAOLccszS'),
          '[^a-zA-Z]', ' ')), ' '),
        CONCAT(' ', p_last, ' ')
      ) = 0 THEN FALSE
      WHEN INSTR(
        CONCAT(' ', LOWER(REGEXP_REPLACE(
          TRANSLATE(full_name,
            'áéíóúüñçÁÉÍÓÚÜÑÇàèìòùâêîôûÀÈÌÒÙÂÊÎÔÛãõÃÕıİğĞşŞçÇöÖüÜæøłÆØŁčćžšŠ',
            'aeiouuncAEIOUUNCaeiouaeiouAEIOUAEIOUaoaoiIgGsScCoOuUaolAOLccszS'),
          '[^a-zA-Z]', ' ')), ' '),
        CONCAT(' ',
               FILTER(SPLIT(REGEXP_REPLACE(p_middle, '[.]', ''), ' +'),
                      x -> LENGTH(x) >= 3)[0],
               ' ')
      ) = 0 THEN FALSE
      ELSE TRUE
    END AS full_name_ok
  FROM profile_with_dom pwd
),
qualifying_groups AS (
  -- Size 3-9, ZERO ORCID-bearing profiles (the Phase 3c twist), sink cap, and
  -- ALL profiles in the group pass full_name_ok.
  SELECT p_first, p_middle, p_last
  FROM profile_consistent
  GROUP BY p_first, p_middle, p_last
  HAVING COUNT(*) BETWEEN 3 AND 9
     AND MAX(works_count) <= 5000
     AND COUNT(CASE WHEN has_orcid THEN 1 END) = 0
     AND COUNT(*) = COUNT(CASE WHEN full_name_ok THEN 1 END)
),
group_profiles AS (
  SELECT pc.*
  FROM profile_consistent pc
  JOIN qualifying_groups g
    ON pc.p_first = g.p_first AND pc.p_middle = g.p_middle AND pc.p_last = g.p_last
),
ranked AS (
  -- Winner = max works_count -> min created_date -> min author_id.
  -- (No ORCID-anchor available; the full_name self-consistency guard above
  --  defends against wrong-winner-selection.)
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY p_first, p_middle, p_last
                            ORDER BY works_count DESC, created_date ASC, author_id ASC) AS rn
  FROM group_profiles
),
winner AS (
  SELECT p_first, p_middle, p_last,
         author_id AS winner_id,
         ARRAY_DISTINCT(CONCAT(lki_ids, aff_ids)) AS winner_inst
  FROM ranked WHERE rn = 1
),
straggler AS (
  SELECT p_first, p_middle, p_last,
         author_id AS straggler_id,
         ARRAY_DISTINCT(CONCAT(lki_ids, aff_ids)) AS straggler_inst
  FROM ranked WHERE rn > 1
),
pair_inst AS (
  SELECT w.p_first, w.p_middle, w.p_last,
         w.winner_id, s.straggler_id,
         (SIZE(ARRAY_INTERSECT(w.winner_inst, s.straggler_inst)) > 0) AS inst_overlap
  FROM winner w
  JOIN straggler s
    ON w.p_first = s.p_first AND w.p_middle = s.p_middle AND w.p_last = s.p_last
),
group_member_coauthors AS (
  -- Per profile, the set of distinct coauthor author_ids on its work_authors rows.
  SELECT wa1.author_id AS profile_id, wa2.author_id AS coauthor_id
  FROM openalex.works.work_authors wa1
  JOIN openalex.works.work_authors wa2
    ON wa2.work_id = wa1.work_id
   AND wa2.author_id <> wa1.author_id
  WHERE wa1.author_id IN (SELECT winner_id    FROM winner)
     OR wa1.author_id IN (SELECT straggler_id FROM straggler)
  GROUP BY wa1.author_id, wa2.author_id
),
pair_coauthor AS (
  -- Coauthors shared between winner and straggler, EXCLUDING any other group member
  -- (otherwise transitive in-group coauthorship inflates the signal artificially).
  SELECT p.p_first, p.p_middle, p.p_last,
         p.winner_id, p.straggler_id,
         COUNT(DISTINCT ac.coauthor_id) AS n_shared_coauthors
  FROM pair_inst p
  JOIN group_member_coauthors ac ON ac.profile_id = p.winner_id
  JOIN group_member_coauthors sc ON sc.profile_id = p.straggler_id
                                AND sc.coauthor_id = ac.coauthor_id
  WHERE ac.coauthor_id NOT IN (SELECT winner_id    FROM winner)
    AND ac.coauthor_id NOT IN (SELECT straggler_id FROM straggler)
  GROUP BY p.p_first, p.p_middle, p.p_last, p.winner_id, p.straggler_id
),
pair_signal AS (
  SELECT pi.p_first, pi.p_middle, pi.p_last,
         pi.winner_id, pi.straggler_id,
         pi.inst_overlap,
         COALESCE(pc.n_shared_coauthors, 0) AS n_shared_coauthors,
         (pi.inst_overlap OR COALESCE(pc.n_shared_coauthors, 0) >= 1) AS any_signal
  FROM pair_inst pi
  LEFT JOIN pair_coauthor pc
    ON pc.winner_id = pi.winner_id AND pc.straggler_id = pi.straggler_id
),
group_signal_ok AS (
  -- Two filters:
  --   (a) every pair has at least one signal (per-pair floor)
  --   (b) at least one pair has inst-overlap OR n_shared_coauthors >= 5
  --       (group-anchor signal floor; Option B; defends against Korean-common-name
  --        and similar romanization-collision classes where weak per-pair signal
  --        across the full group can collapse distinct humans).
  SELECT p_first, p_middle, p_last, winner_id
  FROM pair_signal
  GROUP BY p_first, p_middle, p_last, winner_id
  HAVING COUNT(*) = COUNT(CASE WHEN any_signal THEN 1 END)
     AND COUNT(CASE WHEN inst_overlap OR n_shared_coauthors >= 5 THEN 1 END) >= 1
)
SELECT
  ps.p_first, ps.p_middle, ps.p_last,
  ps.winner_id,
  ps.straggler_id,
  ps.inst_overlap,
  ps.n_shared_coauthors,
  -- Winner metadata (one row per pair carries it)
  wp.full_name      AS winner_full_name,
  wp.works_count    AS winner_works_count,
  wp.cited_by_count AS winner_cited_by_count,
  wp.created_date   AS winner_created_date,
  wp.block_key      AS winner_block_key,
  -- Straggler (loser) metadata
  sp.full_name      AS straggler_full_name,
  sp.works_count    AS straggler_works_count,
  sp.cited_by_count AS straggler_cited_by_count,
  sp.created_date   AS straggler_created_date,
  sp.block_key      AS straggler_block_key
FROM pair_signal ps
JOIN group_signal_ok ok
  ON ok.p_first = ps.p_first AND ok.p_middle = ps.p_middle
 AND ok.p_last  = ps.p_last  AND ok.winner_id = ps.winner_id
JOIN group_profiles wp ON wp.author_id = ps.winner_id
JOIN group_profiles sp ON sp.author_id = ps.straggler_id


### Cell 2: Validation statistics


In [ ]:
%sql
SELECT
  COUNT(DISTINCT winner_id)                                               AS n_groups,
  COUNT(*)                                                                AS n_losers,
  COUNT(DISTINCT winner_id, straggler_id)                                 AS n_pairs,
  AVG(group_size)                                                         AS avg_group_size,
  PERCENTILE_APPROX(group_size, ARRAY(0.5, 0.95, 0.99))                   AS group_size_pctiles,
  SUM(straggler_works_count)                                              AS works_to_rewrite,
  COUNT(CASE WHEN inst_overlap THEN 1 END)                                AS pairs_with_inst,
  COUNT(CASE WHEN n_shared_coauthors >= 1 THEN 1 END)                     AS pairs_with_coauthor,
  COUNT(CASE WHEN inst_overlap AND n_shared_coauthors >= 1 THEN 1 END)    AS pairs_with_both,
  PERCENTILE_APPROX(n_shared_coauthors, ARRAY(0.5, 0.95, 0.99))           AS coauthor_pctiles
FROM (
  SELECT *,
    COUNT(*) OVER (PARTITION BY winner_id) + 1 AS group_size
  FROM openalex.authors.merge_candidates_richname_multi
)


### Cell 3: Spot-check sample (manual review)


In [ ]:
%sql
WITH sampled_winners AS (
  SELECT DISTINCT winner_id
  FROM openalex.authors.merge_candidates_richname_multi
  ORDER BY RAND() LIMIT 30
)
SELECT c.winner_id, c.p_first, c.p_middle, c.p_last,
       c.winner_full_name, c.winner_works_count, c.winner_cited_by_count,
       c.straggler_id, c.straggler_full_name,
       c.straggler_works_count, c.straggler_cited_by_count,
       c.inst_overlap, c.n_shared_coauthors
FROM openalex.authors.merge_candidates_richname_multi c
JOIN sampled_winners s ON s.winner_id = c.winner_id
ORDER BY c.winner_id, c.straggler_works_count DESC


### Cell 4: Create audit log


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.author_merge_log_richname_multi AS
SELECT
  winner_id              AS winner_author_id,
  straggler_id           AS loser_author_id,
  straggler_full_name    AS loser_full_name,
  straggler_works_count  AS loser_works_count,
  straggler_cited_by_count AS loser_cited_by_count,
  straggler_created_date AS loser_created_date,
  p_first, p_middle, p_last,
  inst_overlap,
  n_shared_coauthors,
  current_timestamp()    AS merge_timestamp
FROM openalex.authors.merge_candidates_richname_multi


### Cell 5: Snapshot affected work_ids (capture before MERGE rewrites author_id)


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.author_merge_affected_works_richname_multi AS
SELECT DISTINCT wa.work_id
FROM openalex.works.work_authors wa
JOIN openalex.authors.author_merge_log_richname_multi log
  ON wa.author_id = log.loser_author_id


### Cell 6: Execute the merge — rewrite work_authors.author_id from losers to winners


In [ ]:
%sql
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT loser_author_id, winner_author_id
  FROM openalex.authors.author_merge_log_richname_multi
) AS source
ON target.author_id = source.loser_author_id
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = source.winner_author_id,
    target.updated_at = current_timestamp()


### Cell 7: Queue affected works for reprocessing by UpdateWorkAuthorships

*No DELETE step. Phases 1 / 2 / 3b all use the MERGE-then-soft-zero-via-CreateAuthors approach; this notebook follows that pattern.*


In [ ]:
%sql
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.author_merge_affected_works_richname_multi


### Cell 8: Post-merge verification — losers should drain to 0 works after CreateAuthors


In [ ]:
%sql
WITH log AS (
  SELECT loser_author_id, loser_works_count
  FROM openalex.authors.author_merge_log_richname_multi
),
loser_state AS (
  SELECT l.loser_author_id, l.loser_works_count,
         a.works_count AS current_works_count
  FROM log l
  JOIN openalex.authors.openalex_authors a ON a.id = l.loser_author_id
)
SELECT
  COUNT(*)                                                       AS total_losers,
  COUNT(CASE WHEN current_works_count = 0 THEN 1 END)            AS losers_drained_to_zero,
  COUNT(CASE WHEN current_works_count > 0 THEN 1 END)            AS losers_still_nonzero,
  PERCENTILE_APPROX(current_works_count, ARRAY(0.5, 0.95, 0.99)) AS nonzero_pctiles,
  MAX(current_works_count)                                       AS max_remaining_works
FROM loser_state
